# Regression Model Comparison

Put `train.csv` and `test.csv` in the same directory as this notebook.

- Set `TARGET_COLUMN` to your target column.
- No preprocessing is performed.
- K-fold CV and hyperparameter tuning use **train.csv only**.
- `test.csv` is used only for final evaluation.
- The final cell reports **R², MAE and RMSE** for every model.


In [1]:
# Common imports and data loading
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

TARGET_COLUMN = "target"   # <-- CHANGE THIS

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X = train.drop(columns=[TARGET_COLUMN])
y = train[TARGET_COLUMN]

X_test = test.drop(columns=[TARGET_COLUMN])
y_test = test[TARGET_COLUMN]

CV = KFold(n_splits=2, shuffle=True, random_state=0)

print("Train shape:", train.shape)
print("Test shape :", test.shape)


Train shape: (1600, 21)
Test shape : (400, 21)


## 1. Simple Linear Regression


In [2]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()

# Simple linear regression uses ONE input feature.
# fit_intercept=True estimates the y-axis intercept.
# Cross-validation checks generalization using train.csv only.
cv_scores = cross_val_score(
    regressor, X.iloc[:, [0]], y,
    cv=CV, scoring="r2"
)

regressor.fit(X.iloc[:, [0]], y)

print("Mean 5-Fold CV R2:", cv_scores.mean())
print("CV R2 scores:", cv_scores)


Mean 5-Fold CV R2: 0.04887084207030151
CV R2 scores: [0.02829845 0.06944324]


## 2. Multiple Linear Regression


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV

regressor = LinearRegression()

# fit_intercept: whether to estimate the intercept.
# positive: if True, forces all regression coefficients to be non-negative.
param_grid = {
    "fit_intercept": [True, False],
    "positive": [False, True]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


Best parameters: {'positive': True, 'fit_intercept': False}
Best 2-Fold CV R2: 0.9977158614813088


## 3. Polynomial Regression


In [4]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV

regressor = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear", LinearRegression())
])

# degree: controls polynomial complexity and interaction terms.
# include_bias=False: avoids an extra constant feature because LinearRegression has an intercept.
param_grid = {
    "poly__degree": [2, 3],
    "linear__fit_intercept": [True, False]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


Best parameters: {'poly__degree': 2, 'linear__fit_intercept': False}
Best 2-Fold CV R2: 0.9968032647594638


## 4. Support Vector Regression (SVR)


In [5]:
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV

regressor = SVR(
    kernel="poly",
    C=100,
    epsilon=0.1,
    gamma="scale",
    degree=3,
    coef0=1.0
)

# kernel: defines the function used to model non-linear relationships.
#         Using "poly" instead of "rbf" here (rbf was too slow on this dataset).
# C: controls the penalty for training errors; larger values fit more strictly.
# epsilon: defines the error tolerance tube.
# gamma: controls the influence range of points for the poly/rbf kernels.
# degree: degree of the polynomial kernel (only used when kernel="poly").
# coef0: independent term in the poly kernel; shifts how much high- vs low-degree terms matter.
param_grid = {
    "kernel": ["poly", "linear"],
    "C": [1, 10, 100],
    "epsilon": [0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"],
    "degree": [2, 3],
    "coef0": [0.0, 1.0]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


Best parameters: {'kernel': 'linear', 'gamma': 'scale', 'epsilon': 0.1, 'degree': 2, 'coef0': 0.0, 'C': 10}
Best 2-Fold CV R2: 0.9976702232581509


## 5. Decision Tree Regression


In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = DecisionTreeRegressor(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=0
)

# max_depth: limits tree depth and model complexity.
# min_samples_split: minimum samples required before splitting a node.
# min_samples_leaf: minimum samples allowed in each leaf; larger values can reduce overfitting.
param_grid = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


Best parameters: {'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 5}
Best 2-Fold CV R2: 0.2886727451492708


## 6. Random Forest Regression


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = RandomForestRegressor(
    n_estimators=10,
    random_state=0
)

# n_estimators: number of trees in the forest.
# max_depth: maximum depth of each tree; controls complexity.
# min_samples_split: minimum samples required to split a node.
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


Best parameters: {'n_estimators': 200, 'min_samples_split': 2, 'max_depth': None}
Best 2-Fold CV R2: 0.6803398232049653


## 7. XGBoost Regression


In [8]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    random_state=0,
    objective="reg:squarederror"
)

# n_estimators: number of boosting trees.
# learning_rate: size of each boosting step.
# max_depth: maximum depth of each tree; controls complexity.
# subsample: fraction of training rows used by each tree; can reduce overfitting.
param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8, 1.0]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


Best parameters: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1}
Best 2-Fold CV R2: 0.8971162622762042


## 8. LightGBM Regression


In [9]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,
    num_leaves=31,
    random_state=0,
    verbosity=-1
)

# n_estimators: number of boosting iterations.
# learning_rate: contribution of each boosting tree.
# num_leaves: controls tree complexity; larger values can model more complex patterns.
# max_depth: limits tree depth; -1 means no explicit limit.
param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 10]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


Best parameters: {'num_leaves': 31, 'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.1}
Best 2-Fold CV R2: 0.8521387425148981


## 9. CatBoost Regression


In [10]:
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV

regressor = CatBoostRegressor(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    loss_function="RMSE",
    verbose=False,
    random_seed=0
)

# iterations: number of boosting rounds.
# learning_rate: contribution of each boosting round.
# depth: depth of the trees; higher values increase model complexity.
param_grid = {
    "iterations": [100, 200],
    "learning_rate": [0.05, 0.1],
    "depth": [4, 6, 8]
}

grid = RandomizedSearchCV(
    regressor, param_grid,
    cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
)

grid.fit(X, y)
regressor = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best 2-Fold CV R2:", grid.best_score_)


Best parameters: {'learning_rate': 0.1, 'iterations': 200, 'depth': 6}
Best 2-Fold CV R2: 0.9330545902339601


## Final Test-Set Evaluation


In [11]:
# IMPORTANT:
# Every model is tuned using train.csv only.
# The best configuration is then refit on all of train.csv.
# test.csv remains completely untouched until this cell.

models = {
    "Simple Linear Regression": RandomizedSearchCV(
        LinearRegression(),
        {"fit_intercept": [True, False]},
        cv=CV, scoring="r2", n_iter=3, 
    n_jobs=1
    ),

    "Multiple Linear Regression": RandomizedSearchCV(
        LinearRegression(),
        {"fit_intercept": [True, False], "positive": [False, True]},
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "Polynomial Regression": RandomizedSearchCV(
        Pipeline([
            ("poly", PolynomialFeatures(include_bias=False)),
            ("linear", LinearRegression())
        ]),
        {
            "poly__degree": [2, 3],
            "linear__fit_intercept": [True, False]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "SVR": RandomizedSearchCV(
        SVR(),
        {
            "kernel": ["poly", "linear"],
            "C": [1, 10, 100],
            "epsilon": [0.01, 0.1, 0.2],
            "gamma": ["scale", "auto"],
            "degree": [2, 3],
            "coef0": [0.0, 1.0]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "Decision Tree Regression": RandomizedSearchCV(
        DecisionTreeRegressor(random_state=0),
        {
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "Random Forest Regression": RandomizedSearchCV(
        RandomForestRegressor(random_state=0),
        {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "XGBoost Regression": RandomizedSearchCV(
        XGBRegressor(
            random_state=0,
            objective="reg:squarederror"
        ),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5],
            "subsample": [0.8, 1.0]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "LightGBM Regression": RandomizedSearchCV(
        LGBMRegressor(random_state=0, verbosity=-1),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "num_leaves": [15, 31, 63],
            "max_depth": [-1, 10]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    ),

    "CatBoost Regression": RandomizedSearchCV(
        CatBoostRegressor(
            loss_function="RMSE",
            verbose=False,
            random_seed=0
        ),
        {
            "iterations": [100, 200],
            "learning_rate": [0.05, 0.1],
            "depth": [4, 6, 8]
        },
        cv=CV, scoring="r2", n_iter=3, n_jobs=1
    )
}

results = []

for name, model in models.items():

    # Simple Linear Regression uses only the first feature.
    if name == "Simple Linear Regression":
        model.fit(X.iloc[:, [0]], y)
        predictions = model.predict(X_test.iloc[:, [0]])
    else:
        model.fit(X, y)
        predictions = model.predict(X_test)

    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    results.append([
        name,
        r2,
        mae,
        rmse,
        model.best_params_
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "R2 Score",
        "MAE",
        "RMSE",
        "Best Hyperparameters"
    ]
)

results_df = results_df.sort_values(
    "R2 Score",
    ascending=False
).reset_index(drop=True)

display(results_df)

best_row = results_df.iloc[0]

print("Best model:", best_row["Model"])
print("Best test R2:", best_row["R2 Score"])
print("Selected Hyperparameters:")
print(best_row["Best Hyperparameters"])


,Model,R2 Score,MAE,RMSE,Best Hyperparameters
0,SVR,0.997411,8.015525,10.029248,"{'kernel': 'linear', 'gamma': 'auto', 'epsilon..."
1,Multiple Linear Regression,0.997400,8.038633,10.050384,"{'positive': True, 'fit_intercept': False}"
2,Polynomial Regression,0.997005,8.691294,10.786086,"{'poly__degree': 2, 'linear__fit_intercept': T..."
3,CatBoost Regression,0.917973,43.299750,56.448891,"{'learning_rate': 0.1, 'iterations': 100, 'dep..."
4,XGBoost Regression,0.909118,45.890116,59.417423,"{'subsample': 1.0, 'n_estimators': 200, 'max_d..."
5,LightGBM Regression,0.904212,46.406392,61.000323,"{'num_leaves': 15, 'n_estimators': 200, 'max_d..."
6,Random Forest Regression,0.729359,80.788421,102.535027,"{'n_estimators': 100, 'min_samples_split': 5, ..."
7,Decision Tree Regression,0.363006,127.148797,157.305264,"{'min_samples_split': 2, 'min_samples_leaf': 2..."
8,Simple Linear Regression,0.026090,155.678536,194.506939,{'fit_intercept': True}


Best model: SVR
Best test R2: 0.9974106822654684
Selected Hyperparameters:
{'kernel': 'linear', 'gamma': 'auto', 'epsilon': 0.1, 'degree': 3, 'coef0': 0.0, 'C': 100}
